# Notebook 05: Parameter-Efficient Fine-Tuning (PEFT) with LoRA

In this notebook, we use **LoRA (Low-Rank Adaptation)** to fine-tune the Moirai encoder.
Instead of updating millions of parameters, LoRA freezes the foundation model and injects tiny trainable rank-decomposition matrices into the linear layers.

uv pip install peft

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy
from IPython.display import display


from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder

from heads import *
from models import *
from utils import *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16, 32, 64]

lr_grid=[1e-4]
wd_grid=[0.05, 0.01]

BATCH_SIZE = 256

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

## 2. LoRA Baseline: Mean Pooling on Full Sequence (Context + Mask)

In [4]:
methods = ["LoRA", "DoRA", "AdaLoRA"]
df_mean_pool = pd.DataFrame(index=methods, columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"
df_mean_pool.index.name = "PEFT Method"

for method in methods:
    print(f"\n{'='*40}\nTraining with {method}\n{'='*40}")
    for patch in PATCH_SIZES:
        use_dora = (method == "DoRA")
        use_adalora = (method == "AdaLoRA")
        
        _, test_acc = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": MeanPoolingClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, 
                "remove_last_patch": False, "lora_r": 8,
                "use_dora": use_dora,
                "use_adalora": use_adalora
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_mean_pool.loc[method, patch] = test_acc

display(df_mean_pool.astype(float).round(4))


Training with LoRA
LR=0.0001 | WD=0.05
 Early stopping : 36
Val Loss: 1.0939
LR=0.0001 | WD=0.01
 Early stopping : 38
Val Loss: 1.0631
Acc on Test Set : 0.6452

LR=0.0001 | WD=0.05
 Early stopping : 34
Val Loss: 1.0907
LR=0.0001 | WD=0.01
 Early stopping : 38
Acc on Test Set : 0.6326

LR=0.0001 | WD=0.05
 Early stopping : 36
Val Loss: 1.0732
LR=0.0001 | WD=0.01


 Early stopping : 36
Acc on Test Set : 0.6302

LR=0.0001 | WD=0.05
 Early stopping : 34
Val Loss: 1.0964
LR=0.0001 | WD=0.01
 Early stopping : 32
Acc on Test Set : 0.6290


Training with DoRA
LR=0.0001 | WD=0.05
 Early stopping : 30
Val Loss: 1.1014
LR=0.0001 | WD=0.01
 Early stopping : 42
Val Loss: 1.0483
Acc on Test Set : 0.6302

LR=0.0001 | WD=0.05
 Early stopping : 36
Val Loss: 1.0669
LR=0.0001 | WD=0.01
 Early stopping : 34
Acc on Test Set : 0.6363

LR=0.0001 | WD=0.05
 Early stopping : 35
Val Loss: 1.0797
LR=0.0001 | WD=0.01
 Early stopping : 35
Val Loss: 1.0620
Acc on Test Set : 0.6326

LR=0.0001 | WD=0.05
 Early stopping : 33
Val Loss: 1.1270
LR=0.0001 | WD=0.01
 Early stopping : 38
Val Loss: 1.1022
Acc on Test Set : 0.6354


Training with AdaLoRA
LR=0.0001 | WD=0.05
 Early stopping : 64
Val Loss: 1.0616
LR=0.0001 | WD=0.01
 Early stopping : 62
Val Loss: 1.0496
Acc on Test Set : 0.6285

LR=0.0001 | WD=0.05
 Early stopping : 66
Val Loss: 1.0346
LR=0.0001 | WD=0.01
 Early stoppin

Patch Size,8,16,32,64
PEFT Method,,,,
LoRA,0.6452,0.6326,0.6302,0.6290
DoRA,0.6302,0.6363,0.6326,0.6354
AdaLoRA,0.6285,0.6310,0.6346,0.6253


In [5]:
display(df_mean_pool.astype(float).round(4))

Patch Size,8,16,32,64
PEFT Method,,,,
LoRA,0.6452,0.6326,0.6302,0.6290
DoRA,0.6302,0.6363,0.6326,0.6354
AdaLoRA,0.6285,0.6310,0.6346,0.6253


## 3. LoRA Rank Ablation Study
Testing the impact of the bottleneck rank $r$ on the MHA performance. A higher rank means more trainable parameters.

In [6]:
PATCH = 8
RANKS_TO_TEST = [4, 8, 16, 32, 64, 128] 
methods = ["LoRA", "DoRA", "AdaLoRA"]

df_rank_ablation8 = pd.DataFrame(index=methods, columns=RANKS_TO_TEST)
df_rank_ablation8.index.name = "PEFT Method"
df_rank_ablation8.columns.name = f"Rank 'r' (Patch {PATCH})"

for method in methods:
    for r in RANKS_TO_TEST:
        use_dora = (method == "DoRA")
        use_adalora = (method == "AdaLoRA")
        
        _, test_acc = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": MeanPoolingClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
                "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE, 
                "remove_last_patch": False, "lora_r": r,
                "use_dora": use_dora,
                "use_adalora": use_adalora
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            lr_grid=lr_grid, wd_grid=wd_grid
        )
        df_rank_ablation8.loc[method, r] = test_acc

display(df_rank_ablation8.astype(float).round(4))

LR=0.0001 | WD=0.05
 Early stopping : 41
Val Loss: 1.0690
LR=0.0001 | WD=0.01
 Early stopping : 43
Val Loss: 1.0523
Acc on Test Set : 0.6302

LR=0.0001 | WD=0.05
 Early stopping : 38
Val Loss: 1.0659
LR=0.0001 | WD=0.01
 Early stopping : 35
Acc on Test Set : 0.6436

LR=0.0001 | WD=0.05
 Early stopping : 32
Val Loss: 1.0891
LR=0.0001 | WD=0.01
 Early stopping : 32
Val Loss: 1.0667
Acc on Test Set : 0.6391

LR=0.0001 | WD=0.05
 Early stopping : 33
Val Loss: 1.0743
LR=0.0001 | WD=0.01
 Early stopping : 27
Acc on Test Set : 0.6387

LR=0.0001 | WD=0.05
 Early stopping : 28
Val Loss: 1.0684
LR=0.0001 | WD=0.01
 Early stopping : 28
Val Loss: 1.0594
Acc on Test Set : 0.6488

LR=0.0001 | WD=0.05
 Early stopping : 27
Val Loss: 1.0870
LR=0.0001 | WD=0.01
 Early stopping : 28
Val Loss: 1.0647
Acc on Test Set : 0.6338

LR=0.0001 | WD=0.05
 Early stopping : 39
Val Loss: 1.0645
LR=0.0001 | WD=0.01
 Early stopping : 41
Acc on Test Set : 0.6306

LR=0.0001 | WD=0.05
 Early stopping : 35
Val Loss: 1.0785

Rank 'r' (Patch 8),4,8,16,32,64,128
PEFT Method,,,,,,
LoRA,0.6302,0.6436,0.6391,0.6387,0.6488,0.6338
DoRA,0.6306,0.6403,0.6529,0.6423,0.6468,0.6480
AdaLoRA,0.6294,0.6298,0.6371,0.6221,0.6363,0.6346


In [7]:
display(df_rank_ablation8.astype(float).round(4))

Rank 'r' (Patch 8),4,8,16,32,64,128
PEFT Method,,,,,,
LoRA,0.6302,0.6436,0.6391,0.6387,0.6488,0.6338
DoRA,0.6306,0.6403,0.6529,0.6423,0.6468,0.6480
AdaLoRA,0.6294,0.6298,0.6371,0.6221,0.6363,0.6346


## 4. Advanced LoRA: Multi-Head Attention (Context + Mask)

In [9]:
PATCH_SIZES_TO_TEST = [8, 16] 
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = [f"MHA (Heads={NUM_HEADS} | LoRA r=8)"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, acc_mha = universal_grid_search(
            model_class=LoraHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, 
                "remove_last_patch": False, "lora_r": 64
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(model_names_single[0], mode), patch] = acc_mha

LR=0.0001 | WD=0.05
 Early stopping : 26
Val Loss: 1.0930
LR=0.0001 | WD=0.01
 Early stopping : 23
Acc on Test Set : 0.6156

LR=0.0001 | WD=0.05
 Early stopping : 26
Val Loss: 1.0993
LR=0.0001 | WD=0.01
 Early stopping : 24
Val Loss: 1.0800
Acc on Test Set : 0.6342

LR=0.0001 | WD=0.05
 Early stopping : 25
Val Loss: 1.0954
LR=0.0001 | WD=0.01
 Early stopping : 25
Val Loss: 1.0778
Acc on Test Set : 0.6123

LR=0.0001 | WD=0.05
 Early stopping : 25
Val Loss: 1.0643
LR=0.0001 | WD=0.01
 Early stopping : 24
Val Loss: 1.0371
Acc on Test Set : 0.6172



In [10]:
display(df_adv_single.astype(float).round(4))

Patch Size                                         8       16
Model                     Mode                               
MHA (Heads=16 | LoRA r=8) shared_context       0.6156  0.6123
                          independent_context  0.6342  0.6172